In [1]:
import pandas as pd
import numpy as np
from pathlib import Path
import joblib
from sklearn.preprocessing import MinMaxScaler
from sklearn.ensemble import IsolationForest
from sklearn.model_selection import train_test_split
from imblearn.under_sampling import NearMiss
import warnings

warnings.filterwarnings('ignore')

# ============================================================================
# Project paths
# ============================================================================
PROJECT_ROOT = Path.cwd().parent
DATA_DIR = PROJECT_ROOT / 'data'
RESULTS_DIR = PROJECT_ROOT / 'results'
MODEL_DIR = RESULTS_DIR / 'model'
TABLE_DIR = RESULTS_DIR / 'table'

print("Project paths ready")

print("=" * 80)
print("Step 4: CF Quality Evaluation and Optimal CF Selection (Best-of-N)")
print("=" * 80)

# ============================================================================
# 1. Data Loading
# ============================================================================
print("\n[Step 1] Loading Data")
print("-" * 80)

cf_results = pd.read_csv(DATA_DIR / 'cf_results_4industry.csv')
print(f"CF data loaded: {cf_results.shape}")

selected_features = joblib.load(MODEL_DIR / 'selected_features_final_full.pkl')
model = joblib.load(MODEL_DIR / 'base_model_final_full.pkl')
threshold = joblib.load(MODEL_DIR / 'base_model_threshold_final_full.pkl')

print(f"Analysis target firms: {cf_results['ID'].nunique()}")
print(f"Total CFs to evaluate: {len(cf_results)}")

# ============================================================================
# 2. Reproduce Step2's NearMiss + split procedure (matches Step3B)
# ============================================================================
df_orig = pd.read_csv(DATA_DIR / 'selected_data_for_modeling_full_ratio_clean.csv')
X_full = df_orig[selected_features]
y_full = df_orig['PERF_12M']

nm = NearMiss(version=1, n_neighbors=3)
X_resampled, y_resampled = nm.fit_resample(X_full, y_full)

X_train, X_test, y_train, y_test = train_test_split(
    X_resampled, y_resampled,
    test_size=0.2, random_state=42, stratify=y_resampled
)

# ============================================================================
# 3. Normalization
# ============================================================================
print("\n[Step 2] Data Normalization (for distance calculations)")

scaler = MinMaxScaler()
X_train_scaled = scaler.fit_transform(X_train)

print(f"Scaler fitted on X_train (n={len(X_train)}).")

# ============================================================================
# 4. Isolation Forest — fit on solvent firms only (matches Step3B)
# ============================================================================
solvent_X_raw = df_orig[df_orig['PERF_12M'] == 0][selected_features]
solvent_X_scaled = scaler.transform(solvent_X_raw)

iso_forest = IsolationForest(n_estimators=100, contamination=0.1, random_state=42, n_jobs=-1)
iso_forest.fit(solvent_X_scaled)

print(f"Isolation Forest fitted on solvent firms (n={len(solvent_X_raw)}, scaled space).")

# ============================================================================
# 5. Immutable Features
# ============================================================================
IMMUTABLE = [
    'asset_growth_rate', 'revenue_growth_rate', 'operating_income_growth',
    'net_income_growth', 'equity_growth_rate',
]

missing_immutables = [f for f in IMMUTABLE if f not in selected_features]
if missing_immutables:
    raise ValueError(f"Immutable features not found in feature set: {missing_immutables}")

# ============================================================================
# 6. Physical Validity Check — a second safeguard against DiCE producing
#    numerically extreme values (e.g. genetic-algorithm divergence during
#    optimization), independent of Agent #3's downstream MoE detection layer
#    described in the paper's Section 4.4. Rationale: the paper acknowledges
#    (Section 5.4) that non-negativity and range constraints are currently
#    enforced only at Agent #3, not at the DiCE stage itself; this check adds
#    an upstream gate so an obviously invalid CF (e.g. a debt-repayment
#    coefficient of -7.78e13) never reaches the "optimal" selection in the
#    first place, rather than relying solely on downstream detection.
# ============================================================================

# Reasonable magnitude bounds for ratio/indicator-style features. Chosen to
# be generous (an order of magnitude beyond plausible real-world values) so
# genuine but unusual firms are not falsely excluded — only numerically
# divergent outputs are caught.
PLAUSIBLE_ABS_BOUND = 1e6  # ratio/indicator features should not exceed this in absolute value

def check_physical_validity(cf_vals, feature_names, bound=PLAUSIBLE_ABS_BOUND):
    """
    Flags a CF as physically invalid if any feature value is non-finite
    (inf/NaN) or exceeds a generous magnitude bound, which indicates
    numerical divergence during DiCE's genetic optimization rather than a
    genuine financial scenario.
    """
    for val, name in zip(cf_vals, feature_names):
        if not np.isfinite(val):
            return False, f"{name}=non-finite ({val})"
        if abs(val) > bound:
            return False, f"{name}={val:.2f} exceeds plausible bound ({bound:.0e})"
    return True, None

# ============================================================================
# 7. Single-CF Quality Evaluation Function
# ============================================================================
def check_immutable_violated(original_vals_dict, cf_vals_dict, tol=1e-3):
    """
    Returns True only if an immutable growth-rate feature changed by more
    than a financially meaningful amount (tol=1e-3 accounts for DiCE's
    genetic-algorithm floating-point noise — see Step3B for the diagnostic).
    """
    for f in IMMUTABLE:
        if abs(original_vals_dict[f] - cf_vals_dict[f]) > tol:
            return True
    return False

def evaluate_single_cf(row, scaler, model, threshold, iso_forest):
    """Computes the five quality metrics plus safety flags for a single CF candidate."""
    original_vals = np.array([row[f'Original_{feat}'] for feat in selected_features])
    cf_vals = np.array([row[f'CF_{feat}'] for feat in selected_features])

    # --- Physical validity check runs FIRST, before any scaling/distance math,
    #     since an extreme value could otherwise distort the scaler transform
    #     itself for this row. ---
    is_physically_valid, invalid_reason = check_physical_validity(cf_vals, selected_features)

    if not is_physically_valid:
        # Short-circuit: assign worst-case metrics so this CF is never picked
        # as "optimal" even if downstream code attempts to rank it.
        return {
            'Validity': 0.0,
            'Proximity': np.inf,
            'Sparsity': 0.0,
            'Realism': 0.0,
            'Robustness': 0.0,
            'Immutable_Violated': True,  # conservative: treat as unsafe on both fronts
            'Physically_Valid': False,
            'Invalid_Reason': invalid_reason,
        }

    orig_scaled = scaler.transform(original_vals.reshape(1, -1))[0]
    cf_scaled = scaler.transform(cf_vals.reshape(1, -1))[0]

    pred_prob = model.predict_proba(cf_vals.reshape(1, -1))[0, 1]
    validity = 1.0 if pred_prob < threshold else 0.0

    proximity = np.mean(np.abs(orig_scaled - cf_scaled))

    changes = np.abs(orig_scaled - cf_scaled)
    sparsity = np.mean(changes < 1e-6)

    iso_score = iso_forest.decision_function(cf_scaled.reshape(1, -1))[0]
    realism = (iso_score + 0.5) if iso_score > -0.5 else 0.0

    n_perturb = 10
    noise_level = 0.01
    stable_cnt = 0
    for _ in range(n_perturb):
        noise = np.random.normal(0, noise_level, size=cf_vals.shape)
        perturbed_cf = cf_vals + (noise * (cf_vals + 1e-6))
        p_prob = model.predict_proba(perturbed_cf.reshape(1, -1))[0, 1]
        if p_prob < threshold:
            stable_cnt += 1
    robustness = stable_cnt / n_perturb

    original_dict = {f: row[f'Original_{f}'] for f in IMMUTABLE}
    cf_dict = {f: row[f'CF_{f}'] for f in IMMUTABLE}
    immutable_violated = check_immutable_violated(original_dict, cf_dict)

    return {
        'Validity': validity,
        'Proximity': proximity,
        'Sparsity': sparsity,
        'Realism': realism,
        'Robustness': robustness,
        'Immutable_Violated': immutable_violated,
        'Physically_Valid': True,
        'Invalid_Reason': None,
    }

# ============================================================================
# 8. Run Quality Evaluation
# ============================================================================
print("\n[Step 3] Running Quality Evaluation for All CF Candidates")

evaluated_cfs = []

try:
    from tqdm import tqdm
    iterator = tqdm(cf_results.iterrows(), total=len(cf_results), desc="Evaluating")
except ImportError:
    iterator = cf_results.iterrows()

for idx, row in iterator:
    metrics = evaluate_single_cf(row, scaler, model, threshold, iso_forest)

    res = row.to_dict()
    res.update(metrics)

    if metrics['Physically_Valid']:
        score = (
            (metrics['Validity'] * 5.0) +
            (1.0 - metrics['Proximity']) +
            metrics['Sparsity'] +
            metrics['Realism'] +
            metrics['Robustness']
        ) / 9.0
    else:
        score = -np.inf  # guarantees this CF is never selected as optimal

    res['Quality_Score'] = score
    evaluated_cfs.append(res)

df_evaluated = pd.DataFrame(evaluated_cfs)

print(f"\nEvaluation complete: {len(df_evaluated)} CFs total")
print(f"CFs violating immutable constraint: {df_evaluated['Immutable_Violated'].sum()} "
      f"({df_evaluated['Immutable_Violated'].mean()*100:.1f}%)")
print(f"CFs failing physical validity check: {(~df_evaluated['Physically_Valid']).sum()} "
      f"({(~df_evaluated['Physically_Valid']).mean()*100:.1f}%)")

if (~df_evaluated['Physically_Valid']).sum() > 0:
    print("\n[Physical validity failures — sample reasons]")
    print(df_evaluated[~df_evaluated['Physically_Valid']][['ID', 'CF_Number', 'Invalid_Reason']].head(10).to_string(index=False))

# ============================================================================
# 9. Optimal CF Selection (Best-of-N)
# ============================================================================
print("\n[Step 4] Selecting the Optimal CF per Firm (Top-1)")

# Exclude any CF that violates the immutable constraint OR fails the physical
# validity check from the selection pool before ranking by Quality Score.
def select_best_cf(group):
    valid_pool = group[(~group['Immutable_Violated']) & (group['Physically_Valid'])]
    if valid_pool.empty:
        # Fallback: relax physical-validity requirement before giving up
        # entirely, so a firm is not silently dropped if ALL 4 candidates
        # happen to be physically invalid (logged separately for review).
        valid_pool = group[~group['Immutable_Violated']]
    if valid_pool.empty:
        valid_pool = group
    return valid_pool.loc[valid_pool['Quality_Score'].idxmax()]

df_best = df_evaluated.groupby('ID', group_keys=False).apply(select_best_cf).reset_index(drop=True)

df_final = df_best[df_best['Validity'] == 1.0].copy()

print(f"Firms before selection: {cf_results['ID'].nunique()}")
print(f"Firms after selection: {len(df_final)} (Validity failures excluded)")
print(f"Firms dropped: {cf_results['ID'].nunique() - len(df_final)}")

# Sanity check: confirm no physically invalid CF slipped into the final output
n_invalid_in_final = (~df_final['Physically_Valid']).sum()
print(f"\n[Sanity Check] Physically invalid CFs remaining in final output: {n_invalid_in_final}")
if n_invalid_in_final > 0:
    print("  [WARNING] Some firms had no physically valid candidate among their 4 CFs.")
    print("  These should be reviewed manually before passing to the Agent pipeline.")

# ============================================================================
# 10. Save Results
# ============================================================================
print("\n[Step 5] Saving Results")

output_path = DATA_DIR / 'cf_results_filtered_4industry.csv'
df_final.to_csv(output_path, index=False, encoding='utf-8-sig')
print(f"Optimal CF results saved: {output_path.relative_to(PROJECT_ROOT)}")

details_path = DATA_DIR / 'cf_evaluation_details_4industry.csv'
df_evaluated.to_csv(details_path, index=False, encoding='utf-8-sig')
print(f"Full evaluation details saved: {details_path.relative_to(PROJECT_ROOT)}")

# ============================================================================
# 11. Sample Check
# ============================================================================
if not df_final.empty:
    print("\n[Sample Optimal CF (Top 1)]")
    sample = df_final.iloc[0]
    print(f"ID: {sample['ID']} ({sample.get('SIC_CD_3', 'N/A')})")
    print(f"Quality Score: {sample['Quality_Score']:.4f}")
    print(f"Selected CF Number: #{sample['CF_Number']}")
    print(f"Bankruptcy probability change: {sample['Original_Proba']:.4f} -> {sample['Target_Proba']:.4f}")

    print("\n[Quality Metrics]")
    print(f"- Validity:   {sample['Validity']:.1f}")
    print(f"- Proximity:  {sample['Proximity']:.4f} (lower is better)")
    print(f"- Sparsity:   {sample['Sparsity']:.4f} (higher is better)")
    print(f"- Realism:    {sample['Realism']:.4f} (higher is better)")
    print(f"- Robustness: {sample['Robustness']:.4f} (higher is better)")

    print("\n[Key Changes]")
    for feat in selected_features:
        change = sample[f'Change_{feat}']
        if abs(change) > 0.001:
            print(f"- {feat}: {sample[f'Original_{feat}']:.4f} -> {sample[f'CF_{feat}']:.4f} (Delta: {change:+.4f})")
else:
    print("\n[Warning] No valid CFs found.")

print("\nStep 4 complete. Agent #2 will use this file to generate consulting reports.")

Project paths ready
Step 4: CF Quality Evaluation and Optimal CF Selection (Best-of-N)

[Step 1] Loading Data
--------------------------------------------------------------------------------
CF data loaded: (2443, 191)
Analysis target firms: 640
Total CFs to evaluate: 2443

[Step 2] Data Normalization (for distance calculations)
Scaler fitted on X_train (n=3312).
Isolation Forest fitted on solvent firms (n=145654, scaled space).

[Step 3] Running Quality Evaluation for All CF Candidates


Evaluating: 100%|█████████████████████████████████████████████████████████████████| 2443/2443 [00:15<00:00, 161.67it/s]



Evaluation complete: 2443 CFs total
CFs violating immutable constraint: 0 (0.0%)
CFs failing physical validity check: 0 (0.0%)

[Step 4] Selecting the Optimal CF per Firm (Top-1)
Firms before selection: 640
Firms after selection: 542 (Validity failures excluded)
Firms dropped: 98

[Sanity Check] Physically invalid CFs remaining in final output: 0

[Step 5] Saving Results
Optimal CF results saved: data\cf_results_filtered_4industry.csv
Full evaluation details saved: data\cf_evaluation_details_4industry.csv

[Sample Optimal CF (Top 1)]
ID: 9 (L68)
Quality Score: 0.8270
Selected CF Number: #1
Bankruptcy probability change: 0.9954 -> 0.3700

[Quality Metrics]
- Validity:   1.0
- Proximity:  0.2033 (lower is better)
- Sparsity:   0.2581 (higher is better)
- Realism:    0.4885 (higher is better)
- Robustness: 0.9000 (higher is better)

[Key Changes]
- FN3_3: 94.4200 -> 100.0000 (Delta: +5.5800)
- FN3_6: 238.5000 -> -2363.9873 (Delta: -2602.4873)
- FN3_10: 80.9300 -> 90.1000 (Delta: +9.1700)